# Module 1 — Notebook 03: Pseudobulk DESeq2 + anti-TNF non-response module

**Question.** Within each major cell population of the ulcerative colitis colon (Smillie et al., Cell 2019),
which genes change in inflamed vs healthy tissue — and are the two populations linked to anti-TNF
non-response (inflammatory fibroblasts and OSM+ inflammatory monocytes) expanded and activated in disease?

**Method.** Aggregate raw counts per sample x author cell label (pseudobulk), then run DESeq2
(pydeseq2) per cell label, contrast Inflamed vs Healthy. Sample is the replicate unit
(samples are nested in subjects — a mixed model is the refinement, noted for later).

**Module.** Inflammatory fibroblasts (IL11 / IL13RA2 program) and inflammatory monocytes
(OSM / IL1B axis) were tied to anti-TNF resistance in Smillie 2019 and Martin et al., Cell 2019.
Here: (1) their abundance per sample by condition, (2) their program scores by condition.

Analyses of public datasets; all data credited to original authors.


In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import scanpy as sc
import pydeseq2
print("scanpy", sc.__version__, "| pydeseq2", pydeseq2.__version__)

# Load the QC object from notebook 01 (full gene set + raw counts layer).
# Pseudobulk uses the author's cell labels, so the scVI object is not needed here.
cands = [
    os.path.expanduser("~/Documents/immune-endotypes/data/smillie_uc/adata_qc.h5ad"),
    "../../data/smillie_uc/adata_qc.h5ad",
    "adata_qc.h5ad",
]
path = next(p for p in cands if os.path.exists(p))
adata = sc.read_h5ad(path)
adata.obs = adata.obs.rename(columns={"Health": "condition", "Location": "site", "Cluster": "author_label"})
print(adata)
print(adata.obs["condition"].value_counts())


In [ ]:
# Samples per condition, and the cell labels we will test
print(adata.obs.groupby("condition")["Sample"].nunique())
print(adata.obs["author_label"].nunique(), "author labels")
print(adata.obs["Subject"].nunique(), "subjects |", adata.obs["Sample"].nunique(), "samples")


## 1. Pseudobulk aggregation

Sum raw counts per Sample x author_label (sparse indicator-matrix multiply — no dense copy of the full matrix).
Groups with < 20 cells are dropped.


In [ ]:
import scipy.sparse as sp

counts = adata.layers["counts"].tocsr()
obs = adata.obs
key = obs["Sample"].astype(str) + "|" + obs["author_label"].astype(str)
codes, uniques = pd.factorize(key)
ind = sp.csr_matrix(
    (np.ones(len(codes), dtype=np.float32), (codes, np.arange(len(codes)))),
    shape=(len(uniques), len(codes)),
)
pb = ind @ counts  # samples x genes
pb_df = pd.DataFrame.sparse.from_spmatrix(pb, index=uniques, columns=adata.var_names)

meta_rows = []
for u in uniques:
    sample, label = u.split("|", 1)
    sub = obs[obs["Sample"].astype(str) == sample]
    meta_rows.append({
        "group": u, "Sample": sample, "label": label,
        "Subject": sub["Subject"].iloc[0],
        "condition": sub["condition"].iloc[0],
        "n_cells": int((key == u).sum()),
    })
meta = pd.DataFrame(meta_rows).set_index("group")
meta = meta[meta["n_cells"] >= 20]
pb_df = pb_df.loc[meta.index]
print(pb_df.shape[0], "pseudobulk groups kept")
meta.head()


## 2. DESeq2 per cell label — Inflamed vs Healthy

A label is tested when it has >= 3 Healthy and >= 3 Inflamed samples.
Genes kept per label: count >= 10 in at least 3 samples.


In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

os.makedirs("../results", exist_ok=True)
results = {}
for label, m in meta.groupby("label"):
    conds = m["condition"].value_counts()
    if conds.get("Healthy", 0) < 3 or conds.get("Inflamed", 0) < 3:
        continue
    cdf = pb_df.loc[m.index].sparse.to_dense().astype(int)
    cdf = cdf.loc[:, (cdf >= 10).sum(axis=0) >= 3]
    if cdf.shape[1] < 100:
        continue
    dds = DeseqDataSet(counts=cdf, metadata=m[["condition"]], design_factors="condition")
    dds.deseq2()
    stats = DeseqStats(dds, contrast=["condition", "Inflamed", "Healthy"])
    stats.summary()
    res = stats.results_df.sort_values("padj")
    results[label] = res
    safe = label.replace(" ", "_").replace("+", "pos")
    res.to_csv(f"../results/deseq2_inflamed_vs_healthy_{safe}.csv")
    top = res[res["padj"] < 0.05].head(3).index.tolist()
    print(f"{label:35s} n={len(m):3d}  DE genes (padj<0.05): {(res['padj'] < 0.05).sum():5d}  top: {top}")
print("labels tested:", len(results))


In [ ]:
# Volcano plots for the headline populations
import matplotlib.pyplot as plt

def pick(substr):
    return next((l for l in results if substr.lower() in l.lower()), None)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, substr in zip(axes, ["inflammatory fibroblast", "inflammatory monocyte", "plasma"]):
    label = pick(substr)
    if label is None:
        ax.set_title(f"{substr}: not tested"); continue
    r = results[label].copy()
    r["neglogp"] = -np.log10(r["padj"].clip(lower=1e-300))
    sig = (r["padj"] < 0.05) & (r["log2FoldChange"].abs() > 1)
    ax.scatter(r["log2FoldChange"], r["neglogp"], s=3, c="grey", alpha=0.5)
    ax.scatter(r.loc[sig, "log2FoldChange"], r.loc[sig, "neglogp"], s=5, c="firebrick")
    for g in r[sig].head(6).index:
        ax.annotate(g, (r.loc[g, "log2FoldChange"], r.loc[g, "neglogp"]), fontsize=8)
    ax.axhline(-np.log10(0.05), ls="--", c="k", lw=0.7)
    ax.set_xlabel("log2 fold change (Inflamed / Healthy)")
    ax.set_ylabel("-log10 padj")
    ax.set_title(label)
plt.tight_layout()
plt.savefig("../figures/volcano_key_populations.png", dpi=150)
plt.show()


## 3. Anti-TNF non-response module

Two readouts per sample, by condition:
- **Abundance** — inflammatory fibroblasts as a share of stromal cells; inflammatory monocytes as a share of myeloid cells.
- **Program score** — mean module-gene score (scanpy score_genes) within each population.

Reference biology: Smillie et al., Cell 2019 (inflammatory fibroblasts, IL13RA2+IL11+);
Martin et al., Cell 2019 (GIMATS / OSM axis and anti-TNF resistance).


In [ ]:
fib_label = next(l for l in adata.obs["author_label"].unique() if "inflammatory fibroblast" in l.lower())
mono_label = next(l for l in adata.obs["author_label"].unique() if "inflammatory monocyte" in l.lower())
print("fibroblast label:", fib_label, "| monocyte label:", mono_label)

obs = adata.obs
def abundance(target_label, compartment_l2):
    comp = obs[obs["Major_l2"] == compartment_l2]
    tot = comp.groupby("Sample").size()
    tgt = comp[comp["author_label"] == target_label].groupby("Sample").size()
    frac = (tgt / tot).fillna(0.0)
    cond = obs.drop_duplicates("Sample").set_index("Sample")["condition"]
    return pd.DataFrame({"fraction": frac, "condition": cond.loc[frac.index]})

ab_fib = abundance(fib_label, "Stromal")
ab_mono = abundance(mono_label, "Myeloid")
print(ab_fib.groupby("condition")["fraction"].median())
print(ab_mono.groupby("condition")["fraction"].median())


In [ ]:
# Program scores
fib_genes = ["IL11", "IL13RA2", "IL24", "CHI3L1", "MMP3", "MMP10", "IL6", "CXCL1", "CXCL5"]
mono_genes = ["OSM", "OSMR", "IL1B", "IL6", "TNF", "S100A8", "S100A9"]
for name, genes in [("fib_program", fib_genes), ("mono_program", mono_genes)]:
    present = [g for g in genes if g in adata.var_names]
    print(name, "genes present:", present)
    sc.tl.score_genes(adata, present, score_name=name)

def sample_scores(label, score_col):
    sub = adata.obs[adata.obs["author_label"] == label]
    s = sub.groupby("Sample")[score_col].mean()
    cond = adata.obs.drop_duplicates("Sample").set_index("Sample")["condition"]
    return pd.DataFrame({"score": s, "condition": cond.loc[s.index]})

sc_fib = sample_scores(fib_label, "fib_program")
sc_mono = sample_scores(mono_label, "mono_program")
print(sc_fib.groupby("condition")["score"].median())
print(sc_mono.groupby("condition")["score"].median())


In [ ]:
# Translational summary figure: abundance + program score, by condition
import matplotlib.pyplot as plt

order = ["Healthy", "Non-inflamed", "Inflamed"]
def strip(ax, df, col, title):
    for i, c in enumerate(order):
        v = df[df["condition"] == c][col]
        ax.boxplot(v, positions=[i], widths=0.5, showfliers=False)
        ax.scatter(np.random.normal(i, 0.06, len(v)), v, s=14, alpha=0.7)
    ax.set_xticks(range(3)); ax.set_xticklabels(order)
    ax.set_title(title, fontsize=11)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
strip(axes[0], ab_fib, "fraction", "Inflammatory fibroblasts\n(share of stromal)")
strip(axes[1], sc_fib, "score", "Fibroblast program score\n(IL11 / IL13RA2 axis)")
strip(axes[2], ab_mono, "fraction", "Inflammatory monocytes\n(share of myeloid)")
strip(axes[3], sc_mono, "score", "Monocyte program score\n(OSM / IL1B axis)")
plt.tight_layout()
plt.savefig("../figures/antitnf_module_summary.png", dpi=150)
plt.show()


## Read-out (fill in after running)

- Which labels carry the most DE genes, and do the inflammatory fibroblast / monocyte
  programs sit at the top of their own volcano plots (IL11, IL13RA2, OSM, IL1B)?
- Abundance and program score should both rise Healthy -> Non-inflamed -> Inflamed
  if the module behaves as published. That gradient is the figure a translational
  audience reads first: a disease-activity-linked, drug-resistance-associated cell state.

Next: notebook 04 — cell-cell communication (LIANA) around the OSM-OSMR and IL11 axes;
notebook 05 — the same module in R/Seurat for the cross-platform record.
